# MERT 탐색
MERT-v1-95M 로드 → 오디오 전처리 → feature 출력 확인

In [ ]:
# Colab 전용 설치 (로컬이면 스킵)
# !pip install transformers soundfile torchaudio -q

In [ ]:
import torch
import soundfile as sf
import torchaudio
import numpy as np
from transformers import Wav2Vec2FeatureExtractor, AutoModel

MODEL_ID = "m-a-p/MERT-v1-95M"
MERT_SR = 24000

print(torch.__version__)
print("CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. 모델 & Feature Extractor 로드

In [ ]:
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model = model.to(device).eval()

print("Config:")
print(f"  hidden_size      : {model.config.hidden_size}")
print(f"  num_hidden_layers: {model.config.num_hidden_layers}")
print(f"  sampling_rate    : {processor.sampling_rate}")
print(f"  do_normalize     : {processor.do_normalize}")
print()
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 2. 오디오 로드 & 전처리

In [ ]:
# 프로젝트 루트의 sample.wav 사용 (없으면 경로 수정)
AUDIO_PATH = "../sample.wav"

audio, sr = sf.read(AUDIO_PATH, dtype="float32")
if audio.ndim > 1:
    audio = audio.mean(axis=1)  # stereo → mono

print(f"원본: sr={sr}, shape={audio.shape}, duration={len(audio)/sr:.2f}s")
print(f"min={audio.min():.4f}, max={audio.max():.4f}, std={audio.std():.4f}")

In [ ]:
# MERT_SR(24kHz)로 리샘플
if sr != MERT_SR:
    audio_t = torch.from_numpy(audio).unsqueeze(0)          # (1, T)
    audio_t = torchaudio.functional.resample(audio_t, sr, MERT_SR)
    audio = audio_t.squeeze(0).numpy()
    print(f"리샘플 후: shape={audio.shape}, duration={len(audio)/MERT_SR:.2f}s")
else:
    print("리샘플 불필요")

In [ ]:
# Feature extractor: zero-mean / unit-variance 정규화 + tensor 변환
inputs = processor(audio, sampling_rate=MERT_SR, return_tensors="pt")

print("inputs keys:", list(inputs.keys()))
print("input_values shape:", inputs["input_values"].shape)   # (1, T)
print("mean:", inputs["input_values"].mean().item())
print("std :", inputs["input_values"].std().item())

## 3. MERT Forward → output 구조 확인

In [ ]:
input_values = inputs["input_values"].to(device)

with torch.no_grad():
    outputs = model(input_values=input_values, output_hidden_states=True)

print("outputs 키:", [k for k in dir(outputs) if not k.startswith("_")])
print()
print("last_hidden_state shape:", outputs.last_hidden_state.shape)
print("  → (batch, time_frames, hidden_size)")
print()
print(f"hidden_states 레이어 수: {len(outputs.hidden_states)}")
for i, hs in enumerate(outputs.hidden_states):
    print(f"  layer {i:2d}: {hs.shape}")

## 4. 다운샘플 비율 확인

In [ ]:
T_in  = input_values.shape[-1]
T_out = outputs.last_hidden_state.shape[1]
print(f"입력 샘플 수  : {T_in}")
print(f"출력 프레임 수: {T_out}")
print(f"다운샘플 비율 : {T_in / T_out:.1f}x")
print(f"프레임 주기   : {T_in / T_out / MERT_SR * 1000:.1f} ms/frame")

## 5. Head 입력으로 쓸 embedding 만들기

In [ ]:
# 방법 A: last hidden state mean-pool
emb_last = outputs.last_hidden_state.mean(dim=1)   # (1, 768)
print("A) last layer mean-pool:", emb_last.shape)

# 방법 B: 모든 레이어 weighted sum (학습 가능한 weight 12개)
# hidden_states: tuple of (1, T, 768), 길이 13 (embedding + 12 transformer layers)
all_layers = torch.stack(outputs.hidden_states[1:], dim=0)  # (12, 1, T, 768)
weights = torch.softmax(torch.ones(12), dim=0).to(device)   # 균등 초기화
emb_weighted = (all_layers * weights[:, None, None, None]).sum(0).mean(1)  # (1, 768)
print("B) 12-layer weighted sum mean-pool:", emb_weighted.shape)

print()
print("→ 둘 다 Head 입력 shape은 (B, 768) 동일")

## 6. 배치 처리 확인 (길이가 다른 오디오)

In [ ]:
# 3초짜리 고정 클립 2개를 배치로 처리
CLIP_SEC = 3.0
clip_samples = int(MERT_SR * CLIP_SEC)

def make_clip(audio, clip_samples):
    if len(audio) >= clip_samples:
        return audio[:clip_samples]
    return np.pad(audio, (0, clip_samples - len(audio)))

clip1 = make_clip(audio, clip_samples)
clip2 = make_clip(audio[:clip_samples // 2], clip_samples)  # 짧은 걸 패딩

batch_inputs = processor([clip1, clip2], sampling_rate=MERT_SR, return_tensors="pt")
print("배치 input_values shape:", batch_inputs["input_values"].shape)  # (2, T)

with torch.no_grad():
    batch_out = model(input_values=batch_inputs["input_values"].to(device))

print("배치 last_hidden_state shape:", batch_out.last_hidden_state.shape)  # (2, T', 768)
emb_batch = batch_out.last_hidden_state.mean(dim=1)
print("배치 embedding shape:", emb_batch.shape)  # (2, 768)

## 7. 속도 측정

In [ ]:
import time

x = batch_inputs["input_values"].to(device)

# warmup
with torch.no_grad():
    _ = model(input_values=x)

N = 5
start = time.time()
with torch.no_grad():
    for _ in range(N):
        _ = model(input_values=x)
elapsed = (time.time() - start) / N

print(f"배치 2 × {CLIP_SEC}s 오디오 추론: {elapsed*1000:.1f} ms/iter  (device={device})")